# 01 — Extração e Preparação dos Dados (CRISP-DM, Fase 3)

**Projeto:** Descoberta Gastronômica e Mercado de Alimentação Fora do Lar (GastroMatch) · Documento: [`Preparação_de_Dados.md`](../Preparação_de_Dados.md)

**Entrada:** `dados/brutos/firestore_*.json`, exportação de 25/11/2026 do banco de demonstração `gastromatch-demo` (dados **fictícios**), já sem dados pessoais (`scripts/exportar_gastromatch.py` na raiz do repositório).
**Saída:** `dados/processados/`, com a base consolidada por restaurante, a base longa por experiência, o log de validação e uma planilha Excel.

Etapas do Guia: **Selecionar → Limpar → Construir → Integrar.**

In [1]:
import sys, platform, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# No Colab: clone o repositório e entre na pasta do projeto antes de rodar
#   !git clone https://github.com/gabriellandimdonadel/aprendizado-maquina-udf.git
#   %cd aprendizado-maquina-udf/Projeto_datascience/notebooks
RAIZ = Path("..").resolve()
sys.path.insert(0, str(RAIZ / "src"))
BRUTOS, PROCESSADOS, FIGURAS = RAIZ / "dados" / "brutos", RAIZ / "dados" / "processados", RAIZ / "figuras"
FIGURAS.mkdir(exist_ok=True)
SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "text.parse_math": False, "axes.spines.top": False, "axes.spines.right": False})
FONTE = "Fonte: GastroMatch, base de demonstração gastromatch-demo (dados fictícios), exportada em 25/11/2026"
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | semente {SEMENTE}")

Python 3.11.9 | pandas 3.1.1 | numpy 2.4.6 | semente 42


## 3.1 Selecionar: volume das coleções exportadas

In [2]:
volumes = {p.stem.replace("firestore_", ""): len(json.load(open(p, encoding="utf-8"))) for p in sorted(BRUTOS.glob("firestore_*.json"))}
sel = pd.Series(volumes, name="registros").sort_values(ascending=False).to_frame()
sel["usada na análise"] = sel.index.isin(["restaurantes", "avaliacoes", "menus", "cupons", "resgates"])
print(sel.to_string()); print(f"\nTotal exportado: {sel['registros'].sum():,} registros".replace(",", "."))
print("Coleções técnicas (slugs, documentos, codigosCupom, codigosResgate, admins) e a subcoleção privada de CPF/CNPJ não foram exportadas.")

              registros  usada na análise
avaliacoes         1751              True
menus               748              True
usuarios            232             False
restaurantes        188              True
resgates             66              True
promocoes            35             False
cupons               29              True
eventos              22             False
favoritos             3             False
salvos                1             False

Total exportado: 3.075 registros
Coleções técnicas (slugs, documentos, codigosCupom, codigosResgate, admins) e a subcoleção privada de CPF/CNPJ não foram exportadas.


## 3.2 Limpar + 3.3 Construir + 3.4 Integrar: `src/extracao.py`

O script valida o vocabulário oficial (`app/src/lib/catalogo.js`), confere a consistência das notas, filtra contas inativas, faz o *one-hot* de experiências e ocasiões, deriva os indicadores de cardápio e de cupom e integra tudo por `restauranteId`.

In [3]:
import extracao
base = extracao.main(BRUTOS, PROCESSADOS)
log = pd.read_csv(PROCESSADOS / "log_validacao.csv")
print("\nOcorrências no log por tipo:", log["tipo"].value_counts().to_dict())

    entidade id tipo                                                                      mensagem
restaurantes  -   ok                                               Nenhum campo pessoal na coleção
  avaliacoes  -   ok                                               Nenhum campo pessoal na coleção
       menus  -   ok                                               Nenhum campo pessoal na coleção
      cupons  -   ok                                               Nenhum campo pessoal na coleção
    resgates  -   ok                                               Nenhum campo pessoal na coleção
  avaliacoes  -   ok                                   0 registro(s) com restauranteId inexistente
       menus  -   ok                                   0 registro(s) com restauranteId inexistente
      cupons  -   ok                                   0 registro(s) com restauranteId inexistente
    resgates  -   ok                                   0 registro(s) com restauranteId inexistente
  avaliaco

In [4]:
rest = base.restaurantes
print(f"Base consolidada: {rest.shape[0]} restaurantes × {rest.shape[1]} atributos")
print(f"Base longa (restaurante × experiência): {len(base.vibes_longa)} linhas")
rest[["nome_restaurante", "cozinha", "faixa_preco", "cidade", "vibes_lista", "qtd_vibes", "total_avaliacoes", "nota_media",
      "preco_medio_cardapio", "preco_mediano_principal", "cupons_resgatados", "taxa_conversao_cupom"]].head(8)

Base consolidada: 188 restaurantes × 33 atributos
Base longa (restaurante × experiência): 429 linhas


,nome_restaurante,cozinha,faixa_preco,cidade,vibes_lista,qtd_vibes,total_avaliacoes,nota_media,preco_medio_cardapio,preco_mediano_principal,cupons_resgatados,taxa_conversao_cupom
0,Temaki Mar Foz do Iguaçu,Japonesa,2,Foz do Iguaçu,Sofisticada; Tranquila,2,4,4.7500,45.00,78.50,0,NaN
1,Laboratório Gastronômico,Contemporânea,3,Belo Horizonte,Romântica; Sofisticada,2,3,4.3333,138.75,223.75,0,NaN
2,Rancho da Picanha Caxias do Sul,Churrasco,2,Caxias do Sul,Descontraída; Familiar,2,3,5.0000,51.33,98.00,1,0.0
3,Nonna Rosa,Italiana,3,Campinas,Romântica,1,3,3.0000,64.26,94.40,0,NaN
4,Cedro do Líbano Rio de Janeiro,Árabe,2,Rio de Janeiro,Descontraída; Noturna,2,8,4.1250,29.60,46.50,0,NaN
5,Tempero Caseiro Recife,Brasileira,2,Recife,Descontraída; Tranquila,2,5,4.2000,40.33,53.50,2,0.5
6,Costa Grega,Mediterrânea,3,Florianópolis,Tranquila,1,0,NaN,82.50,127.50,0,NaN
7,Sabor da Roça,Brasileira,2,Belo Horizonte,Tranquila,1,8,3.0000,37.75,55.50,0,NaN


### Variáveis construídas

In [5]:
derivadas = ["qtd_vibes", "qtd_ocasioes", "total_avaliacoes", "nota_media", "qtd_itens_menu", "preco_medio_cardapio",
             "preco_mediano_principal", "cupons_resgatados", "taxa_conversao_cupom"]
desc = rest[derivadas].describe().T.round(2)
desc["ausentes"] = rest[derivadas].isna().sum()
desc

,count,mean,std,min,25%,50%,75%,max,ausentes
qtd_vibes,188.0,2.28,0.75,1.00,2.00,2.00,3.00,3.0,0
qtd_ocasioes,188.0,2.74,1.26,0.00,2.00,3.00,4.00,5.0,0
total_avaliacoes,188.0,6.45,3.40,0.00,3.75,6.50,9.00,14.0,0
nota_media,177.0,3.98,0.55,2.78,3.50,4.00,4.45,5.0,11
qtd_itens_menu,188.0,3.98,0.73,3.00,3.00,4.00,5.00,5.0,0
preco_medio_cardapio,188.0,59.45,29.24,16.00,37.58,53.12,72.20,177.6,0
preco_mediano_principal,186.0,92.57,51.72,27.20,52.29,79.25,117.72,290.0,2
cupons_resgatados,188.0,0.35,1.05,0.00,0.00,0.00,0.00,6.0,0
taxa_conversao_cupom,23.0,0.34,0.20,0.00,0.33,0.33,0.50,0.5,165


Os valores ausentes são **estruturais**: `nota_media` fica vazia nos 11 restaurantes sem avaliação (vazio, não zero, para não puxar a média para baixo), e `taxa_conversao_cupom` só existe para quem teve resgates.

In [6]:
onehot = rest.filter(like="vibe_").sum().sort_values(ascending=False).rename("restaurantes")
print("One-hot de experiências (um restaurante pode ter até 3):"); print(onehot.to_string())
print("\nDistribuição de qtd_vibes:", rest["qtd_vibes"].value_counts().sort_index().to_dict())

One-hot de experiências (um restaurante pode ter até 3):
vibe_familiar        70
vibe_tranquila       66
vibe_descontraida    63
vibe_sofisticada     58
vibe_romantica       51
vibe_alegre          51
vibe_noturna         40
vibe_cafeteria       30

Distribuição de qtd_vibes: {1: 34, 2: 67, 3: 87}


### Validação da planilha-modelo de importação de cardápio (`modelo_cardapio.xlsx`)

As mesmas regras do sistema valem para os cardápios importados em lote: categoria dentro do vocabulário e preço entre R$ 0,01 e R$ 100.000.

In [7]:
import warnings; warnings.filterwarnings("ignore", message="Workbook contains no default style")
modelo = pd.read_excel(BRUTOS / "modelo_cardapio.xlsx").dropna(how="all")
modelo["categoria_valida"] = modelo["Categoria"].isin(extracao.CATEGORIAS_MENU)
modelo["preco_valido"] = modelo["Preço"].between(0.01, 100000)
modelo

,Prato,Categoria,Descrição,Preço,categoria_valida,preco_valido
0,Bruschetta de tomate,Entrada,"Pão italiano, tomate, manjericão e azeite",32.9,True,True
1,Risoto de cogumelos,Prato Principal,"Arroz arbóreo, mix de cogumelos e parmesão",78.0,True,True
2,Petit gâteau,Sobremesa,Bolo quente de chocolate com sorvete de creme,34.5,True,True


## Resultado da Fase 3

- **188 restaurantes ativos, 1.751 avaliações, 748 pratos e 66 resgates** processados, com **0 erros** no log.
- Verificação 1 (`notaMedia = somaNotas ÷ totalAvaliacoes`) e verificação 2 (`totalAvaliacoes` = avaliações contadas): **188/188**.
- Vocabulário: todas as vibes e cozinhas estão dentro das listas oficiais. O código anterior esperava "Cafeteria / Brunch" e uma lista de cozinhas diferente da do sistema, e ambos foram corrigidos.
- LGPD: nenhuma coleção da base analítica contém campos pessoais.
- Saídas em `dados/processados/`: `gastromatch_restaurantes.csv`, `gastromatch_vibes_longa.csv`, `log_validacao.csv` e `gastromatch_analitico.xlsx` (abas `restaurantes_consolidado`, `por_experiencia`, `avaliacoes_resumo`, `log_validacao` e `dicionario`).